# Fine-Tuning QLoRA d'un Modèle Médical (Hackathon IA TechCorp)

Ce notebook implémente le fine-tuning d'un modèle léger (`Phi-3.5-mini-instruct` ou équivalent) sur le dataset médical nettoyé par l'équipe DATA.
Il utilise la méthode **QLoRA** (Quantized Low-Rank Adaptation) pour réduire la consommation de mémoire GPU sur Google Colab.


In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os

# --- Configuration ---
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
# Mettre le chemin vers le dataset nettoyé une fois uploadé sur Colab ou HuggingFace
# Ex: si uploadé sur le Drive ou directement accessible:
DATASET_PATH = "train.json" # Assurez-vous d'avoir uploadé le fichier médical train.json ici
OUTPUT_DIR = "./phi3-medical-finetuned"


In [ ]:
# 1. Chargement du Dataset
print("Chargement du dataset...")
try:
    dataset = load_dataset('json', data_files={'train': DATASET_PATH})
    print(dataset)
except Exception as e:
    print("Erreur lors du chargement du dataset. Assurez-vous d'avoir uploadé 'train.json'.")
    print(e)


In [ ]:
# 2. Configuration de la Quantization (4-bit QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Chargement du modèle de base
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


In [ ]:
# 3. Configuration LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# 4. Configuration de l'entraînement
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=20,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=100,
    warmup_steps=10,
    lr_scheduler_type="constant",
    dataset_text_field="text",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    processing_class=tokenizer,
    args=training_args,
)


In [ ]:
# 5. Lancement de l'entraînement
print("Début de l'entraînement QLoRA...")
trainer.train()

# Sauvegarde du modèle fine-tuné
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Modèle sauvegardé avec succès !")
